Hallazgos o conclusiones que justifiquen las decisiones tomadas posteriormente en el desarrollo de la aplicación.

# 1. Cargar Datos

In [26]:
import pandas as pd # Importación de pandas como pd
import plotly.graph_objects as go # Importación de plotly.graph_objects como go

# Leer los datos del archivo CSV
car_data = pd.read_csv('../vehicles_us.csv')

# 2. Observar Datos Iniciales

In [27]:
print(car_data.info())  # Mostrar información sobre el DataFrame
print(car_data.head(10))  # Mostrar las primeras filas del DataFrame

<class 'pandas.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         51525 non-null  int64  
 1   model_year    47906 non-null  float64
 2   model         51525 non-null  str    
 3   condition     51525 non-null  str    
 4   cylinders     46265 non-null  float64
 5   fuel          51525 non-null  str    
 6   odometer      43633 non-null  float64
 7   transmission  51525 non-null  str    
 8   type          51525 non-null  str    
 9   paint_color   42258 non-null  str    
 10  is_4wd        25572 non-null  float64
 11  date_posted   51525 non-null  str    
 12  days_listed   51525 non-null  int64  
dtypes: float64(4), int64(2), str(7)
memory usage: 7.7 MB
None
   price  model_year           model  condition  cylinders fuel  odometer  \
0   9400      2011.0          bmw x5       good        6.0  gas  145000.0   
1  25500         NaN      ford f-150    

# 3. Preparar Datos

In [28]:
# 1. Convertir la fecha a tipo datetime
car_data['date_posted'] = pd.to_datetime(car_data['date_posted'])

# 2. Convertir la columna 4wd a Booleana (llenando los nulos con False primero)
car_data['is_4wd'] = car_data['is_4wd'].fillna(0).astype(bool)

# 3. Convertir años y cilindros a Enteros (manejando valores nulos con 'Int64')
car_data['model_year'] = car_data['model_year'].astype('Int64')
car_data['cylinders'] = car_data['cylinders'].astype('Int64')

# 4. Convertir columnas de texto repetitivo a Categorías para ahorrar memoria
categorical_cols = ['model', 'condition', 'fuel', 'transmission', 'type', 'paint_color']
for col in categorical_cols:
    car_data[col] = car_data[col].astype('category')

# 5. Verificar los nuevos tipos de datos
print(car_data.info())

<class 'pandas.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   price         51525 non-null  int64         
 1   model_year    47906 non-null  Int64         
 2   model         51525 non-null  category      
 3   condition     51525 non-null  category      
 4   cylinders     46265 non-null  Int64         
 5   fuel          51525 non-null  category      
 6   odometer      43633 non-null  float64       
 7   transmission  51525 non-null  category      
 8   type          51525 non-null  category      
 9   paint_color   42258 non-null  category      
 10  is_4wd        51525 non-null  bool          
 11  date_posted   51525 non-null  datetime64[us]
 12  days_listed   51525 non-null  int64         
dtypes: Int64(2), bool(1), category(6), datetime64[us](1), float64(1), int64(2)
memory usage: 2.8 MB
None


# 4. Manejo de Duplicados

In [29]:
# Contar cuántas filas están completamente duplicadas
total_duplicados = car_data.duplicated().sum()
print(f"Total de filas completamente duplicadas: {total_duplicados}") # 0 Duplicados explícitos detectados


Total de filas completamente duplicadas: 0


In [30]:
# Definir las columnas que identifican a un auto único
columnas_clave = ['model', 'model_year', 'price', 'odometer', 'paint_color']

# Contar cuántos autos sospechosos están repetidos
duplicados_clave = car_data.duplicated(subset=columnas_clave).sum()
print(f"Anuncios sospechosos del mismo auto: {duplicados_clave}")

Anuncios sospechosos del mismo auto: 8397


In [31]:
# Eliminamos los duplicados y reiniciamos el índice
car_data = car_data.drop_duplicates(subset=columnas_clave, keep='first')
car_data = car_data.reset_index(drop=True)

# Confirmamos el nuevo tamaño del dataset
print(f"Nuevo número de filas en el dataset: {car_data.shape[0]}")

Nuevo número de filas en el dataset: 43128


# 5. Manejo de Valores Ausentes

In [32]:
# Detectamos valores nulos en el DataFrame
print(car_data.isnull().sum())

price              0
model_year      3517
model              0
condition          0
cylinders       4388
fuel               0
odometer        7370
transmission       0
type               0
paint_color     8576
is_4wd             0
date_posted        0
days_listed        0
dtype: int64


In [33]:
# Rellenar los colores faltantes con 'unknown'
if 'unknown' not in car_data['paint_color'].cat.categories:
    car_data['paint_color'] = car_data['paint_color'].cat.add_categories('unknown')
car_data['paint_color'] = car_data['paint_color'].fillna('unknown')

# Rellenar 'model_year' con la mediana de año de su propio modelo
car_data['model_year'] = car_data.groupby('model')['model_year'].transform(
    lambda x: x.fillna(round(x.median()) if not pd.isna(x.median()) else 2010)
).astype('Int64')

# Rellenar 'cylinders' con la moda de cilindros de su propio modelo
car_data['cylinders'] = car_data.groupby('model')['cylinders'].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 6))

# Rellenar el kilometraje con la mediana según el año del auto
car_data['odometer'] = car_data.groupby('model_year')['odometer'].transform(lambda x: x.fillna(x.median()))
# Si queda algún nulo suelto (porque el año también era nulo), usamos la mediana general
car_data['odometer'] = car_data['odometer'].fillna(car_data['odometer'].median())

print(car_data.isnull().sum())

price           0
model_year      0
model           0
condition       0
cylinders       0
fuel            0
odometer        0
transmission    0
type            0
paint_color     0
is_4wd          0
date_posted     0
days_listed     0
dtype: int64


# 6. Estadisticas Descriptivas

In [34]:
# Estadísticas descriptivas de variables numéricas
print("Estadísticas Numéricas")
print(car_data[['price', 'model_year', 'cylinders', 'odometer', 'days_listed']].describe().round(2))


Estadísticas Numéricas
           price  model_year  cylinders   odometer  days_listed
count   43128.00     43128.0    43128.0   43128.00     43128.00
mean    11617.88     2009.41       6.06  118282.75        39.60
std      9880.89        6.25       1.65   62447.24        28.22
min         1.00      1908.0        3.0       0.00         0.00
25%      4995.00      2006.0        4.0   78000.00        19.00
50%      8800.00      2010.0        6.0  118622.00        33.00
75%     15500.00      2014.0        8.0  156349.25        53.00
max    375000.00      2019.0       12.0  990000.00       271.00


In [35]:
# Rango de fechas del dataset
print("\nPeriodo de Tiempo de los Anuncios")
print(f"Primer anuncio publicado: {car_data['date_posted'].min()}")
print(f"Último anuncio publicado: {car_data['date_posted'].max()}")



Periodo de Tiempo de los Anuncios
Primer anuncio publicado: 2018-05-01 00:00:00
Último anuncio publicado: 2019-04-19 00:00:00


In [36]:
# Precio promedio y cantidad de autos por condición
print("\nPrecio promedio por Condición del Auto")
print(car_data.groupby('condition')['price'].agg(['mean', 'count']).round(2).sort_values(by='mean', ascending=False))


Precio promedio por Condición del Auto
               mean  count
condition                 
new        26208.46    141
like new   16413.41   4194
excellent  12748.11  20789
good        9669.21  16315
salvage     4056.32    113
fair        3357.04   1576


In [37]:
# Filtrar precios poco realistas o extremadamente altos
car_data_clean = car_data[(car_data['price'] >= 500) & (car_data['price'] <= 100000)]

# Filtrar autos excesivamente antiguos para el análisis común
car_data_clean = car_data_clean[car_data_clean['model_year'] >= 1980]

print(f"Filas originales: {len(car_data)}")
print(f"Filas tras limpiar anomalías: {len(car_data_clean)}")

Filas originales: 43128
Filas tras limpiar anomalías: 42324


# 7. Visualizaciones

In [38]:
# Creo un histograma para visualizar la distribución de precios de venta de los autos
fig_hist = go.Figure()

fig_hist.add_trace(go.Histogram(
    x=car_data_clean['price'],
    nbinsx=50,
    marker_color='#1f77b4',
    opacity=0.75
))

fig_hist.update_layout(
    title='<b>¿Cómo se distribuyen los precios de venta de los autos?</b>',
    xaxis_title='Precio ($)',
    yaxis_title='Cantidad de Anuncios',
    template='plotly_white',
    bargap=0.05
)

fig_hist.show()

In [39]:
# Tomamos una muestra aleatoria para optimizar el rendimiento interactivo de Plotly
df_sample = car_data_clean.sample(n=min(2000, len(car_data_clean)), random_state=42)

fig_scatter = go.Figure()

fig_scatter.add_trace(go.Scatter(
    x=df_sample['odometer'],
    y=df_sample['price'],
    mode='markers',
    marker=dict(
        size=7,
        color=df_sample['model_year'],  # El color representa el año del modelo
        colorscale='Viridis',          # Escala de colores estéticamente agradable
        showscale=True,
        colorbar=dict(title="Año del Modelo")
    ),
    text=df_sample['model']  # Al pasar el cursor verás de qué modelo de auto se trata
))

fig_scatter.update_layout(
    title='<b>Relación entre Kilometraje (Odometer) y Precio</b>',
    xaxis_title='Kilometraje Recorrido',
    yaxis_title='Precio ($)',
    template='plotly_white'
)

fig_scatter.show()


In [40]:
fig_box = go.Figure()

# Definimos un orden lógico de desgaste para el eje X
condiciones_ordenadas = ['new', 'like new', 'excellent', 'good', 'fair', 'salvage']

for condicion in condiciones_ordenadas:
    # Filtramos el dataset para obtener solo los autos de la condición actual
    df_cond = car_data_clean[car_data_clean['condition'] == condicion]
    
    fig_box.add_trace(go.Box(
        y=df_cond['price'],
        name=condicion,
        boxpoints='outliers',  # Muestra los puntos atípicos fuera del cuerpo de la caja
        marker=dict(size=4)
    ))

fig_box.update_layout(
    title='<b>Distribución y Rangos de Precios según la Condición del Auto</b>',
    xaxis_title='Condición Declarada',
    yaxis_title='Precio ($)',
    template='plotly_white',
    showlegend=False  # Ocultamos la leyenda ya que el eje X tiene los nombres claros
)

fig_box.show()


# 8. Conclusiones

Hallazgos y Conclusiones del EDA
A partir del análisis exploratorio y las visualizaciones interactivas desarrolladas en Plotly, se extrajeron las siguientes conclusiones clave que justifican el diseño y la lógica de la aplicación:
1. Segmentación de Búsqueda por Precio: El histograma de precios muestra un fuerte sesgo a la derecha, concentrando la mayor cantidad de anuncios entre los $3,000 y $10,000. 
2. Lógica de Alertas por Kilometraje: El gráfico de dispersión confirma una devaluación exponencial: al superar las 100k unidades de kilometraje, el techo de los precios cae drásticamente por debajo de los $40k.
3. Variable Crítica de Entrada (Condición): El gráfico de cajas (boxplot) demuestra de forma contundente cómo la mediana del precio desciende escalonadamente según el desgaste, comprimiendo los precios de categorías como fair y salvage al fondo de la escala.
4. Optimización del Rendimiento Interactivo: El alto volumen del dataset original ralentizaba el renderizado de gráficos de dispersión con miles de puntos interactivos.